# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MahmooudEssam/FlyRank-Intern/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — connect to the warehouse

*Runs in Colab (token from Secrets as `HF_TOKEN` or `hf_token`) or locally (env var `HF_TOKEN`, or a saved `huggingface-cli login`). The token is never printed.*

To stay under Hugging Face rate limits I only read **two mid-panel month partitions** (Feb and Mar 2026), never the full 79M-row table and never the `_sample` (June = the sealed test month).

In [ ]:
import os, sys, subprocess

try:
    import duckdb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb
import pandas as pd
import numpy as np

def get_hf_token():
    """Colab Secrets -> env var -> saved huggingface-cli login. Never printed."""
    try:
        from google.colab import userdata
        for name in ("HF_TOKEN", "hf_token"):
            try:
                tok = userdata.get(name)
                if tok:
                    return tok
            except Exception:
                pass
    except ImportError:
        pass
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    try:
        from huggingface_hub import get_token
        return get_token()
    except Exception:
        return None

hf_token = get_hf_token()
assert hf_token, "No Hugging Face token found - add HF_TOKEN to Colab Secrets (or env var)."

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
FEAT_FILES = f"{FACT}/month=2026-02/*.parquet"   # feature window lives here
TGT_FILES  = f"{FACT}/month=2026-03/*.parquet"   # target window lives here

# Sanity check: row counts + date range of the two partitions I use
display(con.sql(f"""
    SELECT month, COUNT(*) AS rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS clients
    FROM read_parquet(['{FEAT_FILES}', '{TGT_FILES}'])
    GROUP BY month ORDER BY month
""").df())

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Ranking / scoring.** The output is an ordered list: *which page-2 pages should an editor review first?* Under the hood I will train a **binary classifier** that gives each candidate page a probability of reaching page 1 in the next window, and I sort pages by that probability. Editors have limited hours, so the order of the list matters more than a yes/no for every page.


**My one-paragraph frame:**
> For **content editors**, deciding **which page-2 pages (avg position 11–20) to optimize first**, I will build a **ranked review queue** from **the warehouse daily facts (GSC + GA4 signals)**, scoring **the chance a page reaches page 1 (position ≤ 10) in the next 28 days**, measured by **precision@50**. A wrong call costs **editor hours spent on pages that would not move (or risk to pages that were already moving)**. A plain rule isn't enough because **closeness to page 1, demand, recent direction, CTR and site effects interact** (section 5). I will claim only **observed, decision-support** results: not that an edit *causes* a page to rise.

In [ ]:
# Setup + sanity check is in section 0 above; nothing extra needed for the task type.
print('Task type: ranking / scoring (binary classifier probability -> sorted review queue)')

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Decision point:** 2026-03-01.

| Window | Dates | Used for |
|---|---|---|
| Feature window | 2026-02-01 → 2026-02-28 (28 days) | everything the model may see |
| Target window | 2026-03-01 → 2026-03-28 (28 days) | the label only |

The two windows **do not overlap**, so no future information enters the features.

**Who is a candidate** (the pages the editor would choose between): in the feature window, `gsc_impressions ≥ 100` **and** impression-weighted avg position in **(10, 20]** — page 2.

**Target `reached_page1`** = 1 if, in the target window, the page has **≥ 30 impressions** and impression-weighted avg position **≤ 10**; else 0.

- This is an **observed future outcome**, not a rule copied from the product.
- It is still a **proxy**: what I really want is "would reach page 1 *if an editor improved it*". The data has no edit log, so I can only observe "reached page 1 next month" — natural momentum. I say this out loud in every write-up.
- Candidates with < 30 impressions in March count as 0 (they lost visibility, or tracking gaps). I report how many there are so it is not hidden.
- Weighted position = `SUM(gsc_sum_position) / SUM(gsc_impressions)` — a page with 1,000 impressions counts more than a day with 1.
- Thresholds (100, 30, 10, 20) are **policy choices**; I will test alternatives in W04/W06.
- Feb→Mar is my **development** pair. June 2026 (the last month) stays sealed as the final test.

In [ ]:
# Build the slice: one row per page (client x content) at the decision point 2026-03-01.
SLICE_SQL = f"""
WITH feat AS (
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions)                                         AS impr_f,
        SUM(gsc_clicks)                                              AS clicks_f,
        SUM(gsc_sum_position)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS pos_f,
        COUNT(*) FILTER (WHERE gsc_impressions > 0)                  AS days_with_impr_f,
        -- direction inside the feature window: first half vs second half of Feb
        SUM(gsc_sum_position) FILTER (WHERE report_date <  DATE '2026-02-15')::DOUBLE
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date <  DATE '2026-02-15'), 0) AS pos_f_early,
        SUM(gsc_sum_position) FILTER (WHERE report_date >= DATE '2026-02-15')::DOUBLE
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-02-15'), 0) AS pos_f_late,
        BOOL_OR(ga4_data_available)                                  AS has_ga4_f,
        SUM(ga4_sessions)         FILTER (WHERE ga4_data_available)  AS sessions_f,
        SUM(ga4_engaged_sessions) FILTER (WHERE ga4_data_available)  AS engaged_sessions_f
    FROM read_parquet('{FEAT_FILES}')
    WHERE gsc_data_available
      AND report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
    GROUP BY client_hash_id, content_hash_id
),
tgt AS (
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions)                                         AS impr_t,
        SUM(gsc_sum_position)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS pos_t
    FROM read_parquet('{TGT_FILES}')
    WHERE gsc_data_available
      AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-28'
    GROUP BY client_hash_id, content_hash_id
)
SELECT f.*,
       COALESCE(t.impr_t, 0) AS impr_t,
       t.pos_t,
       CASE WHEN t.impr_t >= 30 AND t.pos_t <= 10 THEN 1 ELSE 0 END AS reached_page1
FROM feat f
LEFT JOIN tgt t USING (client_hash_id, content_hash_id)
WHERE f.impr_f >= 100 AND f.pos_f > 10 AND f.pos_f <= 20
"""
df = con.sql(SLICE_SQL).df()

# Derived feature-window signals (feature window only)
df["ctr_f"] = df["clicks_f"] / df["impr_f"]                      # a fraction, 0.01 = 1%
df["pos_change_f"] = df["pos_f_early"] - df["pos_f_late"]        # > 0 = moving UP during Feb

n = len(df)
print(f"Candidate page-2 pages: {n:,}  from {df['client_hash_id'].nunique()} clients")
print(f"reached_page1 = 1: {df['reached_page1'].sum():,}  ({df['reached_page1'].mean():.1%} base rate)")
print(f"Candidates with < 30 impressions in target window (counted as 0): "
      f"{(df['impr_t'] < 30).sum():,}  ({(df['impr_t'] < 30).mean():.1%})")
display(df["reached_page1"].value_counts().rename("pages").to_frame())

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50** — of the top 50 pages my score sends to editors, how many actually reached page 1 in the next 28 days? 50 is a realistic monthly review capacity, and it matches how the list is used (people read the top, not the whole list). Average precision is my secondary metric (quality of the whole ranking).

**What "good" means** — two bars, both computed below on today's data:
1. **Random pick** = the base rate. Anything at or near it is worthless.
2. **The simple rule** "closest to page 1 first" (sort by feature-window position, ascending). This is the bar my model must beat.

"Good" = the model's precision@50 is **clearly above the rule's**, measured on **clients the model never saw in training** (client holdout), and the gain holds on the sealed June test month. If it doesn't beat the rule, I report that honestly.

In [ ]:
def precision_at_k(frame, score_col, k, ascending=False, label="reached_page1"):
    top = frame.sort_values(score_col, ascending=ascending).head(k)
    return top[label].mean()

base_rate = df["reached_page1"].mean()
rows = []
for k in (20, 50, 100):
    rows.append({
        "K": k,
        "random (base rate)": base_rate,
        "rule: closest to page 1": precision_at_k(df, "pos_f", k, ascending=True),
        "rule: most impressions": precision_at_k(df, "impr_f", k, ascending=False),
    })
bars = pd.DataFrame(rows).set_index("K")
display(bars.style.format("{:.1%}"))
print("The model (W05) must beat the 'closest to page 1' row at K=50, on held-out clients.")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (of one client) at the decision point 2026-03-01**: its signals summed over the 28 days before, plus whether it reached page 1 in the 28 days after.

Column roles:

| Role | Columns |
|---|---|
| Join / group keys (never features) | `client_hash_id`, `content_hash_id` — `client_hash_id` is used for the grouped train/test split |
| Features (feature window only) | `impr_f`, `clicks_f`, `ctr_f`, `pos_f`, `pos_f_early`, `pos_f_late`, `pos_change_f`, `days_with_impr_f`, `has_ga4_f`, `sessions_f`, `engaged_sessions_f` |
| Label | `reached_page1` |
| Target-window values (leakage — **never features**) | `impr_t`, `pos_t` |

GA4 columns are only counted where `ga4_data_available` is TRUE, so a missing GA4 history shows up as `NaN` / `has_ga4_f = False`, not as fake zeros.

In [ ]:
FEATURES = ["impr_f", "clicks_f", "ctr_f", "pos_f", "pos_f_early", "pos_f_late", "pos_change_f",
            "days_with_impr_f", "has_ga4_f", "sessions_f", "engaged_sessions_f"]
KEYS = ["client_hash_id", "content_hash_id"]
LABEL = "reached_page1"
LEAKY = ["impr_t", "pos_t"]

print("shape:", df.shape)
# Grain probe: each (client, content) must appear exactly once
dupes = df.groupby(KEYS).size().loc[lambda s: s > 1]
print("duplicate rows at the grain:", len(dupes))
assert len(dupes) == 0

display(df[KEYS + FEATURES + [LABEL]].head(10))
display(df[FEATURES].describe().T.round(2))

# Client concentration -> why the split must be grouped by client
per_client = df.groupby("client_hash_id")[LABEL].agg(pages="size", positives="sum", rate="mean")
print(f"Largest client holds {per_client['pages'].max() / len(df):.1%} of candidate pages")
display(per_client.sort_values("pages", ascending=False).head(10).style.format({"rate": "{:.1%}"}))

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

An if-statement can only look at one or two signals with fixed cut-offs ("position < 13 and impressions > 500"). Whether a page-2 page climbs depends on several signals **at the same time**:

- **How close** it is (position 11 vs 19).
- **How much demand** it has (impressions) — more data, but also more competitive queries.
- **Recent direction** inside the feature window (`pos_change_f`): a page at 15 that is climbing may beat a page at 11 that is sliding.
- **Click behaviour** (CTR) and **consistency** (days with impressions).
- **Site effects** — some clients' pages move more than others (section 4 table).

The tables below show the chance of reaching page 1 changes across *combinations* of signals, not along one line — that is the pattern a learned score can pick up and a single threshold can't. This is a hypothesis to test, not a result: if the rule wins in W04/W05, the rule is the answer and I'll say so.

In [ ]:
d = df.copy()
d["pos_bucket"] = pd.cut(d["pos_f"], [10, 12, 14, 16, 18, 20])
d["impr_bucket"] = pd.qcut(d["impr_f"].rank(method="first"), 3, labels=["low", "mid", "high"])
d["direction"] = pd.cut(d["pos_change_f"], [-np.inf, -1, 1, np.inf],
                        labels=["sliding (>1 pos down)", "flat", "climbing (>1 pos up)"])

print("Chance of reaching page 1 - position bucket x impressions tercile")
display(d.pivot_table(index="pos_bucket", columns="impr_bucket", values=LABEL,
                      aggfunc="mean", observed=False).style.format("{:.1%}"))

print("Chance of reaching page 1 - position bucket x direction during Feb")
display(d.pivot_table(index="pos_bucket", columns="direction", values=LABEL,
                      aggfunc="mean", observed=False).style.format("{:.1%}"))

print("Pages per cell (small cells = noisy rates)")
display(d.pivot_table(index="pos_bucket", columns="direction", values=LABEL,
                      aggfunc="size", observed=False))

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.